<a href="https://colab.research.google.com/github/Trannganne/face-mask-tracking-system/blob/feature%2Ftracking_timer/notebooks/training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Chạy trong Google Colab:



In [ ]:
# Mount drive và giải nén
from google.colab import drive
drive.mount('/content/drive')
!unzip -q "/content/drive/MyDrive/Face_Mask_Project/Dataset_Mask.zip" -d /content/dataset

In [ ]:
!pip install torch torchvision opencv-python pillow matplotlib seaborn scikit-learn
!pip install deep-sort-realtime
!apt-get install -y libsndfile1
!pip install soundfile
!pip install facenet-pytorch  # Để detect face

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import cv2
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report, precision_recall_fscore_support
import time
import os
from collections import defaultdict
import soundfile as sf
import io

class MaskDetectionCNN(nn.Module):
  def __init__(self, num_classes=3):
    super(MaskDetectionCNN, self).__init__()

    #Block 1: 3-> 32
    self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
    self.bn1 = nn.BatchNorm2d(32)
    self.pool1=nn.MaxPool2d(2,2)
    self.dropout1=nn.Dropout2d(0.25)

    #Block 2: 32 -> 64
    self.conv2=nn.Conv2d(32, 64, kernel_size=3, padding=1)
    self.bn2=nn.BatchNorm2d(64)
    self.pool2=nn.MaxPool2d(2,2)
    self.dropout2=nn.Dropout2d(0.25)

    # Block 3: 64 -> 128
    self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
    self.bn3 = nn.BatchNorm2d(128)
    self.pool3 = nn.MaxPool2d(2, 2)
    self.dropout3 = nn.Dropout2d(0.3)

    # Block 4: 128 -> 256
    self.conv4 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
    self.bn4 = nn.BatchNorm2d(256)
    self.pool4 = nn.MaxPool2d(2, 2)
    self.dropout4 = nn.Dropout2d(0.3)

    # Fully Connected Layers
    self.fc1 = nn.Linear(256 * 8 * 8, 512)
    self.dropout_fc1 = nn.Dropout(0.5)

    self.fc2 = nn.Linear(512, 128)
    self.dropout_fc2 = nn.Dropout(0.5)

    self.fc3 = nn.Linear(128, num_classes)

  def forward(self, x):
    #Block 1
    x=self.conv1(x)
    x=self.bn1(x)
    x=torch.relu(x)
    x=self.pool1(x)
    x=self.dropout1(x)

    # Block 2
    x = self.conv2(x)
    x = self.bn2(x)
    x = torch.relu(x)
    x = self.pool2(x)
    x = self.dropout2(x)

    # Block 3
    x = self.conv3(x)
    x = self.bn3(x)
    x = torch.relu(x)
    x = self.pool3(x)
    x = self.dropout3(x)

    # Block 4
    x = self.conv4(x)
    x = self.bn4(x)
    x = torch.relu(x)
    x = self.pool4(x)
    x = self.dropout4(x)

    #Flatten
    x=x.view(x.size(0), -1)

    #Fully Connected Layers
    x=self.fc1(x)
    x=torch.relu(x)
    x=self.dropout_fc1(x)

    x=self.fc2(x)
    x=torch.relu(x)
    x=self.dropout_fc2(x)

    x=self.fc3(x)

    return x



🏗️ KIẾN TRÚC MODEL: Custom CNN
    
    INPUT: (batch, 3, 128, 128) - Ảnh RGB 128x128
    
    BLOCK 1: Convolutional Block 1
    - Conv2d: 3 → 32 channels, kernel=3, padding=1
    - BatchNorm2d: 32 channels
    - ReLU activation
    - MaxPool2d: kernel=2, stride=2
    - Dropout: p=0.25
    → Output: (batch, 32, 64, 64)
    BLOCK 2: Convolutional Block 2
    - Conv2d: 32 → 64 channels, kernel=3, padding=1
    - BatchNorm2d: 64 channels
    - ReLU activation
    - MaxPool2d: kernel=2, stride=2
    - Dropout: p=0.25
    → Output: (batch, 64, 32, 32)
    
    BLOCK 3: Convolutional Block 3
    - Conv2d: 64 → 128 channels, kernel=3, padding=1
    - BatchNorm2d: 128 channels
    - ReLU activation
    - MaxPool2d: kernel=2, stride=2
    - Dropout: p=0.3
    → Output: (batch, 128, 16, 16)
    
    BLOCK 4: Convolutional Block 4
    - Conv2d: 128 → 256 channels, kernel=3, padding=1
    - BatchNorm2d: 256 channels
    - ReLU activation
    - MaxPool2d: kernel=2, stride=2
    - Dropout: p=0.3
    → Output: (batch, 256, 8, 8)
    FLATTEN: (batch, 256, 8, 8) → (batch, 16384)
    
    FULLY CONNECTED LAYERS:
    - FC1: 16384 → 512 nodes
      - ReLU activation
      - Dropout: p=0.5
    
    - FC2: 512 → 128 nodes
      - ReLU activation
      - Dropout: p=0.5
    
    - FC3 (Output): 128 → 3 nodes (3 classes)
    
    OUTPUT: (batch, 3) - Logits cho 3 classes:
      0: Đeo đúng cách (with_mask)
      1: Đeo sai cách (incorrect_mask)
      2: Không đeo (without_mask)
    
    TỔNG SỐ THAM SỐ: ~8.7 triệu parameters

============= Bước 3: HÀM TỐI ƯU HÓA ====================

 HÀM TRAIN MODEL
    
    Optimizer: Adam
    - Learning rate: 0.001
    - Weight decay: 1e-4 (L2 regularization)
    
    Loss Function: CrossEntropyLoss
    
    Learning Rate Scheduler: ReduceLROnPlateau
    - Giảm LR khi val_loss không giảm sau 5 epochs
    - Factor: 0.5 (giảm 50%)
    
    Early Stopping:
    - Dừng nếu val_loss không cải thiện sau 10 epochs

In [ ]:
def train_model(model, train_loader, val_loader, num_epochs, learning_rate=0.001):
    device=torch.device('cuda'if torch.cuda.is_available() else 'cpu')
    model.to(device)

    #Optimizer
    optimizer=optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-4)

    #Loss function
    criterion=nn.CrossEntropyLoss()

    #Learning rate scheduler
    scheduler=optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5, verbose=True)

    #Early stopping
    best_val_loss=float('inf')
    patience=10
    patience_counter=0

    history={
        'train_loss':[],
        'val_loss':[],
        'train_acc':[],
        'val_acc':[]
    }

    for epoch in range(num_epochs):
      # TRAINING
      model.train()
      train_loss=0.0
      train_correct=0
      train_total=0

      for images, labels in train_loader:
        images=images.to(device)
        labels=labels.to(device)

        #Forward pass
        outputs=model(images)
        loss=criterion(outputs, labels)

        #Backward pass

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        #Statistics
        train_loss+=loss.item()
        _, predicted=torch.max(outputs.data, 1)
        train_total+=labels.size(0)
        train_correct+=(predicted==labels).sum().item()

      train_loss/=len(train_loader)
      train_acc=100*train_correct/train_total

      # VALIDATION
      model.eval()
      val_loss=0.0
      val_correct=0
      val_total=0

      with torch.no_grad():
        for images, labels in val_loader:
          images, labels = images.to(device), labels.to(device)

          outputs = model(images)
          loss = criterion(outputs, labels)

          val_loss += loss.item()
          _, predicted = torch.max(outputs.data, 1)
          val_total += labels.size(0)
          val_correct += (predicted == labels).sum().item()

      val_loss/=len(val_loader)
      val_acc=100*val_correct/val_total

      #Learning rate scheduler
      scheduler.step(val_loss)

      #Lưu lịch sử
      history['train_loss'].append(train_loss)
      history['train_acc'].append(train_acc)
      history['val_loss'].append(val_loss)
      history['val_acc'].append(val_acc)

      print(f'Epoch {epoch+1}/{num_epochs}')
      print(f'Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%')
      print(f'Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%')

      #Early stopping
      if(val_loss<best_val_loss):
        best_val_loss=val_loss
        patience_counter=0
        # Lưu model tốt nhất
        torch.save(model.state_dict(),'best_mask_model.pth')
        print('Đã lưu model tốt nhất!')
      else:
        patience_counter+=1
        if(patience_counter>=patience):
          print(f'Dừng sớm tại epoch {epoch+1}')
          break

    return model, history

BƯỚC 4: HÀM ĐÁNH GIÁ

 Metrics:
    
    - Precision: TP / (TP + FP)
    
    - Recall: TP / (TP + FN)
    
    - F1-Score: 2 * (Precision * Recall) / (Precision + Recall)
    
    - Confusion Matrix


In [ ]:
def evaluate_model(model, test_loader, class_names=['Đeo đúng', 'Đeo sai', 'Không đeo']):
  device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
  model.eval()

  all_preds = []
  all_labels = []

  with torch.no_grad():
      for images, labels in test_loader:
          images = images.to(device)
          outputs = model(images)
          _, predicted = torch.max(outputs.data, 1)

          all_preds.extend(predicted.cpu().numpy())
          all_labels.extend(labels.numpy())

  # Calculate metrics
  precision, recall, f1, support = precision_recall_fscore_support(
      all_labels, all_preds, average=None
    )

  # IN metrics
  print("\n"+"="*60)
  print("Đánh giá model")
  print("="*60)

  for i, class_name in enumerate(class_names):
    print(f"{class_name}:")
    print(f"Precision: {precision[i]:.4f}")
    print(f"Recall: {recall[i]:.4f}")
    print(f"F1-Score: {f1[i]:.4f}")
    print(f"Support: {support[i]}")

  # Metrics tổng quan
  overall_precision = np.mean(precision)
  overall_recall = np.mean(recall)
  overall_f1 = np.mean(f1)

  print("\n"+"="*60)
  print(f"Precision: {overall_precision:.4f}")
  print(f"Recall: {overall_recall:.4f}")
  print(f"F1-Score: {overall_f1:.4f}")
  print("="*60)

  # Confusion matrix
  cm = confusion_matrix(all_labels, all_preds)
  print("Confusion Matrix:")
  print(cm)

  # Báo cáo phân loại
  print("\n Báo cáo")
  print(classification_report(all_labels, all_preds, target_names=class_names))

  # Ma trận nhầm lẫn
  cm=confusion_matrix(all_labels, all_preds)

  plt.figure(figsize=(10,8))
  sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
  plt.title('Confusion Matrix', fontsize=16, fontweight='bold')
  plt.xlabel('Predicted Label', fontsize=12)
  plt.ylabel('True Label', fontsize=12)
  plt.tight_layout()
  plt.savefig('confusion_matrix.png',dpi=300, bbox_inches='tight')
  plt.show()

  return precision, recall, f1

BƯỚC 5: DEEPSORT TRACKING + CẢNH BÁO
Chức năng:
    - Track từng người bằng DeepSORT
    - Đếm thời gian không đeo khẩu trang
    - Phát cảnh báo nếu > 20s

In [ ]:
class PersonTracker:
  def __init__(self):
        from deep_sort_realtime.deepsort_tracker import DeepSort

        self.tracker = DeepSort(
            max_age=30,
            n_init=3,
            nn_budget=100,
            max_iou_distance=0.7
        )

        # Lưu trạng thái từng người
        self.person_states = defaultdict(lambda: {
            'start_time': None,
            'total_no_mask_time': 0,
            'last_state': None,
            'warned': False
        })

        # Tạo âm thanh cảnh báo (440Hz - nốt La)
        self.alert_sound = self.generate_alert_sound()

  def generate_alert_sound(self, duration=1.0, frequency=440):
        """Tạo âm thanh cảnh báo"""
        sample_rate = 22050
        t = np.linspace(0, duration, int(sample_rate * duration))
        wave = 0.5 * np.sin(2 * np.pi * frequency * t)
        return wave

  def play_alert(self):
        """Phát âm thanh cảnh báo"""
        try:
            sf.write('alert.wav', self.alert_sound, 22050)
            print("🚨 CẢNH BÁO: Phát hiện người không đeo khẩu trang quá 20 giây!")
        except:
            print("🚨 CẢNH BÁO: Không thể phát âm thanh!")

  def update(self, detections, current_time):
        """
        Cập nhật tracking và kiểm tra cảnh báo

        Args:
            detections: List of (bbox, confidence, class_id)
            current_time: Thời gian hiện tại

        Returns:
            List of tracked objects với cảnh báo
        """
        # Convert to DeepSORT format
        ds_detections = []
        for bbox, conf, class_id in detections:
            x1, y1, x2, y2 = bbox
            ds_detections.append(([x1, y1, x2-x1, y2-y1], conf, class_id))

        # Update tracker
        tracks = self.tracker.update_tracks(ds_detections, frame=None)

        results = []

        for track in tracks:
            if not track.is_confirmed():
                continue

            track_id = track.track_id
            bbox = track.to_ltrb()
            class_id = track.get_det_class()

            state = self.person_states[track_id]

            # Class: 0=sai, 1=đúng, 2=Không đeo
            if class_id == 1:  # Đeo khẩu trang
                if state['start_time'] is None:
                    state['start_time'] = current_time

                mask_duration = current_time - state['start_time']
                state['total_no_mask_time'] = mask_duration

                # Cảnh báo nếu > 20s
                if mask_duration > 20 and not state['warned']:
                    self.play_alert()
                    state['warned'] = True

                warning = mask_duration > 20
            else:
                # Reset nếu đeo khẩu trang
                state['start_time'] = None
                state['total_no_mask_time'] = 0
                state['warned'] = False
                mask_duration = 0
                warning = False

            state['last_state'] = class_id

            results.append({
                'track_id': track_id,
                'bbox': bbox,
                'class_id': class_id,
                'mask_time': mask_duration,
                'warning': warning
            })

        return results


BƯỚC 6: INFERENCE VIDEO

In [1]:
#XỬ LÝ VIDEO VỚI TRACKING
from facenet_pytorch import MTCNN

def process_video(model, video_path, output_path='output_video.mp4'):
  device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
  model = model.to(device)
  model.eval()

  #Khởi tạo Face Detector (MTCNN)
  mtcnn = MTCNN(keep_all=True, device=device)

  # Khởi tạo tracker load âm thanh
  tracker = PersonTracker()
  tracker.alert_sound, _ = sf.read('/content/drive/MyDrive/Face_Mask_Project/mixkit-sound-alert-in-hall-1006.wav')

  # Mở video
  cap = cv2.VideoCapture(video_path)

    # Cấu hình video output
  fourcc = cv2.VideoWriter_fourcc(*'mp4v')
  fps = int(cap.get(cv2.CAP_PROP_FPS))
  width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
  height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
  out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

  # Transform
  transform = transforms.Compose([
      transforms.Resize((128, 128)),
      transforms.ToTensor(),
      transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
  ])

  class_names = ['mask_weared_incorrect', 'with_mask', 'without_mask']
  colors = [(0, 165, 255),(0, 255, 0), (0, 0, 255)]  # Green, Orange, Red

  frame_count = 0
  start_time = time.time()

  print("🎬 Đang xử lý video...")

  while cap.isOpened():
      ret, frame = cap.read()
      if not ret:
          break

      img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

      # Bước 1: Tìm khuôn mặt
      boxes, probs = mtcnn.detect(img_rgb)

      frame_count += 1
      current_time = time.time() - start_time

        # TODO: Detect faces (dùng MTCNN hoặc face detector)
        # Giả lập: Giả sử đã detect được faces
      detections = []  # Format: [(bbox, conf, class_id), ...]
      if boxes is not None:
          for box, prob in zip(boxes, probs):
              if prob<0.9: continue # Loại bỏ nhận diện yếu

              #Cắt khuôn mặt
              x1, y1, x2, y2 = map(int, box)
              face = img_rgb[y1:y2, x1:x2]

              if face.size==0:continue

              # Bước 2: Dự đoán khẩu trang bằng CNN
              face_pil=Image.fromarray(face)
              face_tensor=transform(face_pil).unsqueeze(0).to(device)

              with torch.no_grad():
                  output=model(face_tensor)
                  class_id=torch.argmax(output).item()
                  conf=torch.softmax(output, dim=1)[0][class_id].item()

              detections.append(([x1, y1, x2, y2], conf, class_id))

      # Bước 3: Tracking & cảnh báo
      tracked_objects = tracker.update(detections, current_time)

        # Vẽ kết quả
      for obj in tracked_objects:
          track_id = obj['track_id']
          x1, y1, x2, y2 = map(int, obj['bbox'])
          class_id = obj['class_id']
          no_mask_time = obj['no_mask_time']
          warning = obj['warning']

          color = colors[class_id]
          label = f"ID:{track_id} {class_names[class_id]}"

            # Vẽ bbox
          cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)

            # Vẽ label
          cv2.putText(frame, label, (x1, y1-10),
                     cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

             # Hiển thị thời gian không đeo
          if class_id == 2:
              time_text = f"No mask: {no_mask_time:.1f}s"
              cv2.putText(frame, time_text, (x1, y2+20),
                         cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)

           # Cảnh báo
          if warning:
              cv2.putText(frame, "WARNING!", (x1, y2+40),
                         cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

      out.write(frame)

      if frame_count % 30 == 0:
          print(f"  Processed {frame_count} frames...")

  cap.release()
  out.release()
  print(f"✅ Đã lưu video: {output_path}")

SyntaxError: invalid syntax (688571522.py, line 2)

In [ ]:
# CHUẨN BỊ DỮ LIỆU
from torchvision.datasets import ImageFolder

def prepare_data(data_dir='/content/dataset'):
    """
    Giả sử cấu trúc: /content/dataset/train/class_0, /content/dataset/train/class_1...
    """
    transform = transforms.Compose([
        transforms.Resize((128, 128)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    train_set = ImageFolder(os.path.join(data_dir, 'train'), transform=transform)
    val_set = ImageFolder(os.path.join(data_dir, 'val'), transform=transform)
    test_set = ImageFolder(os.path.join(data_dir, 'test'), transform=transform)

    train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_set, batch_size=32)
    test_loader = DataLoader(test_set, batch_size=32)

    return train_loader, val_loader, test_loader

BƯỚC 7: MAIN - CHẠY TRÊN COLAB

In [ ]:
def main():

    print("="*70)
    print("CHƯƠNG TRÌNH NHẬN DIỆN KHẨU TRANG + DEEPSORT TRACKING")
    print("="*70)

    # ===== 1. KHỞI TẠO MODEL =====
    print("\n📦 Khởi tạo model...")
    model = MaskDetectionCNN(num_classes=3)

    # In thông tin model
    print(f"\n🏗️ Kiến trúc model:")
    print(model)

    total_params = sum(p.numel() for p in model.parameters())
    print(f"\n📊 Tổng số parameters: {total_params:,}")

    # ===== 2. TRAIN MODEL  =====
    # TODO: Chuẩn bị dataset và train
    train_loader, val_loader, test_loader = prepare_data()
    model, history = train_model(model, train_loader, val_loader, num_epochs=50)

    # ===== 3. ĐÁNH GIÁ MODEL =====
    # TODO: Load trained model và evaluate
    #model.load_state_dict(torch.load('best_mask_model.pth'))
    #evaluate_model(model, test_loader)

    # ===== 4. INFERENCE VIDEO =====
    # Sau khi train xong,
    #process_video(model, 'input_video.mp4', 'output_video.mp4')

    print("\n Hoàn thành!")

if __name__ == "__main__":
    main()